# CS-317: Reinforcement Learning Lab 6
## Advanced Planning by Dynamic Programming
## Muhammad Sikandar Hussain 502808
**BS AI-1**

This notebook provides the emergency-response environment and a few helper functions. Complete every `# TODO` cell, run all cells top to bottom without errors, and answer the written questions in the markdown cells provided for them.

The notebook must execute from beginning to end without manual modification. Use fixed random seeds where appropriate, label all figures, and report the experimental settings.

NumPy, Pandas and Matplotlib may be used. Reinforcement-learning libraries (e.g. `gymnasium`, `stable-baselines3`) and ready-made policy-iteration or value-iteration implementations are not permitted. Do not use Q-learning, SARSA or other model-free algorithms: every planning algorithm must explicitly use the transition probabilities.

In [2]:
import time
import itertools
from math import comb

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.precision", 3)

---
## Provided Starter Code

Run the cells in this section as-is. You do not need to modify them, but read `build_mdp` carefully: it is the full transition model, and you will need to understand the state indexing and the layout of `P` and `R` for every task below.

- `P_table[state][action]` lists the outcomes `(probability, next_state, reward)` of every feasible action.
- `P[s, a, s2]` and `R[s, a, s2]` are the same model as arrays, indexed by state number (`STATE_INDEX[(u, r, c, f)]`). Rows of infeasible actions are all zero.
- `FEASIBLE[s]` lists the feasible actions of state number `s`, and `FEASIBLE_MASK` is the same information as a boolean array of shape `(54, 4)`.

**Counting Bellman backups.** Throughout this lab, one Bellman backup is the computation of $\sum_{s'} P(s'|s,a)\,[\,R(s,a,s') + \gamma V(s')\,]$ for **one** state–action pair. A policy-evaluation sweep of a deterministic policy therefore costs 54 backups, while an improvement step or a value-iteration sweep costs `N_SA` backups (one per feasible state–action pair).

In [4]:
# ---------------------------------------------------------------
# Emergency Response Resource Allocation
#
# State s = (u, r, c, f)
#   u : urgent backlog       0, 1, 2
#   r : routine backlog      0, 1, 2
#   c : available units      0, 1, 2
#   f : fatigue              0 = normal, 1 = high
#
# Actions: URGENT, ROUTINE, BALANCED, RECOVER   (index 0, 1, 2, 3)
# Reward : R(s, a, s') = -C(s, a, s')   (negative operational cost)
# ---------------------------------------------------------------

ACTIONS = ["URGENT", "ROUTINE", "BALANCED", "RECOVER"]
STATES = [(u, r, c, f) for u in range(3) for r in range(3) for c in range(3) for f in range(2)]
STATE_INDEX = {s: i for i, s in enumerate(STATES)}
N_STATES, N_ACTIONS = len(STATES), len(ACTIONS)


def feasible_actions(state):
    """Returns the list of action indices that may be taken in `state`."""
    u, r, c, f = state
    acts = []
    if u > 0 and c >= 1:
        acts.append(0)                      # URGENT
    if r > 0 and c >= 1:
        acts.append(1)                      # ROUTINE
    if u > 0 and r > 0 and c >= 2:
        acts.append(2)                      # BALANCED
    acts.append(3)                          # RECOVER
    return acts


FEASIBLE = [feasible_actions(s) for s in STATES]          # FEASIBLE[s_idx] -> list of actions
FEASIBLE_MASK = np.zeros((N_STATES, N_ACTIONS), dtype=bool)
for i, acts in enumerate(FEASIBLE):
    FEASIBLE_MASK[i, acts] = True
N_SA = int(FEASIBLE_MASK.sum())                            # number of feasible (s, a) pairs


def _binomial(n, p):
    return [(k, comb(n, k) * p**k * (1 - p)**(n - k)) for k in range(n + 1)]


def build_mdp(p_urgent=0.35, p_routine=0.45, reliability=0.85, fatigued_reliability=0.55,
              p_fatigue=0.30, p_drain=0.40, p_recover_fatigue=0.80,
              w_urgent=6.0, w_routine=1.5, w_overflow=15.0, w_fatigue=2.0, w_unit=0.5):
    """Environment generator. Returns the transition table

        P_table[state][action] = list of (probability, next_state, reward)

    for every state and every FEASIBLE action. One decision step happens in
    three stages:

    1. Service. URGENT sends min(c, u) units to urgent incidents, ROUTINE sends
       min(c, r) units to routine incidents, BALANCED sends one unit to each,
       RECOVER sends none. Each unit resolves its incident with probability
       `reliability` (normal fatigue) or `fatigued_reliability` (high fatigue).
    2. Resources. After a service action, capacity drops by one unit with
       probability `p_drain`, and normal fatigue becomes high with probability
       `p_fatigue` (high fatigue stays high). RECOVER restores one unit of
       capacity and returns fatigue to normal with probability `p_recover_fatigue`.
    3. Arrivals. One new urgent incident arrives with probability `p_urgent` and
       one new routine incident with probability `p_routine`. A backlog cannot
       exceed 2; any excess incident is an overflow.

    Cost C = w_urgent*u' + w_routine*r' + w_overflow*overflow + w_fatigue*f' + w_unit*units_sent
    and the reward is -C. The cost weights define the reward function.
    """
    P_table = {}
    for (u, r, c, f) in STATES:
        P_table[(u, r, c, f)] = {}
        q = reliability if f == 0 else fatigued_reliability
        for a in feasible_actions((u, r, c, f)):
            if a == 0:   ku, kr = min(c, u), 0
            elif a == 1: ku, kr = 0, min(c, r)
            elif a == 2: ku, kr = 1, 1
            else:        ku, kr = 0, 0
            units = ku + kr

            if a == 3:
                cap = [(min(2, c + 1), 1.0)]
                fat = [(0, p_recover_fatigue), (1, 1 - p_recover_fatigue)] if f == 1 else [(0, 1.0)]
            else:
                cap = [(max(0, c - 1), p_drain), (c, 1 - p_drain)]
                fat = [(1, 1.0)] if f == 1 else [(1, p_fatigue), (0, 1 - p_fatigue)]

            outcomes = {}
            for (su, pu), (sr, pr), (cn, pc), (fn, pf), (au, pa), (ar, pb) in itertools.product(
                    _binomial(ku, q), _binomial(kr, q), cap, fat,
                    [(1, p_urgent), (0, 1 - p_urgent)], [(1, p_routine), (0, 1 - p_routine)]):
                prob = pu * pr * pc * pf * pa * pb
                if prob == 0:
                    continue
                u1, r1 = u - su + au, r - sr + ar
                un, rn = min(2, u1), min(2, r1)
                overflow = (u1 - un) + (r1 - rn)
                cost = (w_urgent * un + w_routine * rn + w_overflow * overflow
                        + w_fatigue * fn + w_unit * units)
                key = ((un, rn, cn, fn), -cost)
                outcomes[key] = outcomes.get(key, 0.0) + prob
            P_table[(u, r, c, f)][a] = [(p, s2, rew) for (s2, rew), p in outcomes.items()]
    return P_table


def to_arrays(P_table):
    """Converts the transition table to arrays used by the planning algorithms.

    P[s, a, s2] : transition probability (all zeros for an infeasible action)
    R[s, a, s2] : expected reward of that transition
    """
    P = np.zeros((N_STATES, N_ACTIONS, N_STATES))
    RW = np.zeros((N_STATES, N_ACTIONS, N_STATES))
    for state, acts in P_table.items():
        s = STATE_INDEX[state]
        for a, outcomes in acts.items():
            for p, s2, rew in outcomes:
                P[s, a, STATE_INDEX[s2]] += p
                RW[s, a, STATE_INDEX[s2]] += p * rew
    R = np.divide(RW, P, out=np.zeros_like(RW), where=P > 0)
    return P, R


P_table = build_mdp()
P, R = to_arrays(P_table)
print(f"{N_STATES} states, {N_ACTIONS} actions, {N_SA} feasible state-action pairs")
print("Outcomes of URGENT in state (2, 0, 2, 0):")
for p, s2, rew in sorted(P_table[(2, 0, 2, 0)][0], key=lambda o: -o[0])[:5]:
    print(f"   next state {s2}: probability {p:.3f}, reward {rew:+.1f}")
print("   ...")

54 states, 4 actions, 110 feasible state-action pairs
Outcomes of URGENT in state (2, 0, 2, 0):
   next state (0, 0, 2, 0): probability 0.108, reward -1.0
   next state (1, 0, 2, 0): probability 0.097, reward -7.0
   next state (0, 1, 2, 0): probability 0.089, reward -2.5
   next state (1, 1, 2, 0): probability 0.079, reward -8.5
   next state (0, 0, 1, 0): probability 0.072, reward -1.0
   ...


In [5]:
# ---------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------

def actions_to_policy(actions):
    """Converts an array of action indices, shape (N_STATES,), into a policy
    matrix pi[s, a] of shape (N_STATES, N_ACTIONS)."""
    pi = np.zeros((N_STATES, N_ACTIONS))
    pi[np.arange(N_STATES), actions] = 1.0
    return pi


# Initial-state distribution used for every "final return" and every simulation:
# uniform over all 54 states.
INIT_DIST = np.full(N_STATES, 1.0 / N_STATES)

# Settings for the altered environment of Part D (all other parameters, including
# the cost weights, stay at their defaults).
ALTERED_SETTINGS = dict(p_urgent=0.60, reliability=0.70, fatigued_reliability=0.40)


def simulate(P, R, policy, n_episodes=1000, horizon=100, gamma=0.95, seed=42, init_dist=INIT_DIST):
    """Vectorized Monte Carlo simulation using the supplied transition model."""
    rng = np.random.default_rng(seed)
    out = {k: np.zeros(n_episodes) for k in ["return", "urgent", "exhausted", "fatigued"]}
    states = rng.choice(N_STATES, size=n_episodes, p=init_dist)
    discount = 1.0
    for t in range(horizon):
        u = np.array([STATES[s][0] for s in states])
        c = np.array([STATES[s][2] for s in states])
        f = np.array([STATES[s][3] for s in states])
        out["urgent"] += u / horizon
        out["exhausted"] += (c == 0) / horizon
        out["fatigued"] += (f == 1) / horizon
        actions = np.array([rng.choice(N_ACTIONS, p=policy[s]) for s in states])
        next_states = np.empty(n_episodes, dtype=int)
        rewards = np.empty(n_episodes)
        for s in np.unique(states):
            idx = np.flatnonzero(states == s)
            for a in np.unique(actions[idx]):
                chosen = idx[actions[idx] == a]
                next_states[chosen] = rng.choice(N_STATES, size=len(chosen), p=P[s, a])
                rewards[chosen] = R[s, a, next_states[chosen]]
        out["return"] += discount * rewards
        discount *= gamma
        states = next_states
    return out


print("Feasible actions in (1, 1, 2, 0):", [ACTIONS[a] for a in FEASIBLE[STATE_INDEX[(1, 1, 2, 0)]]])

Feasible actions in (1, 1, 2, 0): ['URGENT', 'ROUTINE', 'BALANCED', 'RECOVER']


---
## Part A: Understand and Evaluate the MDP

### Task A1: Inspect the Environment

Inspect the state space, the available actions, the transition probabilities and the rewards. For the four states below, report the feasible actions and explain the operational constraints.

In [7]:
check_states = [(2, 0, 2, 0), (1, 2, 1, 1), (2, 2, 0, 1), (0, 0, 2, 0)]
for state in check_states:
    print(f"\nState {state}: feasible actions = {[ACTIONS[a] for a in FEASIBLE[STATE_INDEX[state]]]} ")
    for a in FEASIBLE[STATE_INDEX[state]]:
        outcomes = sorted(P_table[state][a], key=lambda x: -x[0])[:3]
        print(f"  {ACTIONS[a]} (top outcomes):", [(s2, round(prob, 3), rew) for prob, s2, rew in outcomes])



State (2, 0, 2, 0): feasible actions = ['URGENT', 'RECOVER'] 
  URGENT (top outcomes): [((0, 0, 2, 0), 0.108, -1.0), ((1, 0, 2, 0), 0.097, -7.0), ((0, 1, 2, 0), 0.089, -2.5)]
  RECOVER (top outcomes): [((2, 0, 2, 0), 0.358, -12.0), ((2, 1, 2, 0), 0.293, -13.5), ((2, 0, 2, 0), 0.193, -27.0)]

State (1, 2, 1, 1): feasible actions = ['URGENT', 'ROUTINE', 'RECOVER'] 
  URGENT (top outcomes): [((1, 2, 1, 1), 0.16, -11.5), ((1, 2, 1, 1), 0.131, -26.5), ((0, 2, 1, 1), 0.118, -5.5)]
  ROUTINE (top outcomes): [((1, 2, 1, 1), 0.193, -11.5), ((1, 2, 0, 1), 0.129, -11.5), ((1, 1, 1, 1), 0.118, -10.0)]
  RECOVER (top outcomes): [((1, 2, 2, 0), 0.286, -9.0), ((1, 2, 2, 0), 0.234, -24.0), ((2, 2, 2, 0), 0.154, -15.0)]

State (2, 2, 0, 1): feasible actions = ['RECOVER'] 
  RECOVER (top outcomes): [((2, 2, 1, 0), 0.388, -30.0), ((2, 2, 1, 0), 0.286, -15.0), ((2, 2, 1, 0), 0.126, -45.0)]

State (0, 0, 2, 0): feasible actions = ['RECOVER'] 
  RECOVER (top outcomes): [((0, 0, 2, 0), 0.358, -0.0), ((0, 1,

| State $(u, r, c, f)$ | Feasible actions | Operational interpretation |
|---|---|---|
| (2, 0, 2, 0) | URGENT, RECOVER | Two urgent jobs and two units; routine service and balanced dispatch are infeasible because routine backlog is zero. |
| (1, 2, 1, 1) | URGENT, ROUTINE, RECOVER | One unit forces a single category to be served; fatigue is high, reducing service reliability. |
| (2, 2, 0, 1) | RECOVER | No units are available, so service actions are infeasible; recovery restores capacity and may reduce fatigue. |
| (0, 0, 2, 0) | RECOVER | No incidents are waiting; only recovery is feasible, preserving/restoring capacity as new arrivals may occur. |

In [9]:
def validate_mdp(P_table, states, actions, tol=1e-9):
    state_set = set(states)
    ok = True
    n_actions = len(actions)
    for state in states:
        listed = P_table.get(state, {})
        expected = set(feasible_actions(state))
        if set(listed) != expected:
            print(f"Feasibility mismatch at {state}: {set(listed)} != {expected}")
            ok = False
        for a, outcomes in listed.items():
            probs = [x[0] for x in outcomes]
            if any(p < -tol for p in probs) or not np.isclose(sum(probs), 1.0, atol=tol):
                print(f"Invalid probabilities at {(state, a)}: sum={sum(probs)}")
                ok = False
            if any(s2 not in state_set for _, s2, _ in outcomes):
                print(f"Invalid successor state at {(state, a)}")
                ok = False
            if not 0 <= a < n_actions:
                print(f"Invalid action index {a}")
                ok = False
    print(f"MDP validation: {'PASS' if ok else 'FAIL'} ({len(states)} states, {sum(map(len, P_table.values()))} feasible state-action rows)")
    return ok

validate_mdp(P_table, STATES, ACTIONS)


MDP validation: PASS (54 states, 110 feasible state-action rows)


### Task A2: Evaluate Competing Operational Policies

Implement the three strategies as policy matrices of shape `(54, 4)` that give zero probability to infeasible actions:

- **Urgent-first:** give priority to urgent incidents whenever feasible.
- **Recovery-aware:** prefer recovery when fatigue is high.
- **Uniform random:** choose uniformly among feasible actions.

Then evaluate them using your policy-evaluation implementation from the previous lab.

In [11]:
# Deterministic priorities: urgent first -> routine -> recover; recovery-aware chooses
# recover whenever fatigued, otherwise follows urgent-first. Random is uniform feasible.
def urgent_action(s):
    acts = FEASIBLE[s]
    return 0 if 0 in acts else (1 if 1 in acts else 3)
urgent_actions = np.array([urgent_action(s) for s in range(N_STATES)])
urgent_first_policy = actions_to_policy(urgent_actions)
recovery_actions = np.array([3 if STATES[s][3] == 1 else urgent_action(s) for s in range(N_STATES)])
recovery_aware_policy = actions_to_policy(recovery_actions)
uniform_random_policy = np.zeros((N_STATES, N_ACTIONS))
for s, acts in enumerate(FEASIBLE):
    uniform_random_policy[s, acts] = 1.0 / len(acts)
print("Rules: urgent-first uses URGENT, then ROUTINE, then RECOVER; recovery-aware uses RECOVER at high fatigue and urgent-first otherwise; random is uniform over feasible actions.")


Rules: urgent-first uses URGENT, then ROUTINE, then RECOVER; recovery-aware uses RECOVER at high fatigue and urgent-first otherwise; random is uniform over feasible actions.


In [12]:
def evaluate_policy(P, R, policy, gamma=0.95, theta=1e-8):
    V = np.zeros(N_STATES)
    sweeps = 0
    while True:
        old = V.copy()
        q = np.sum(P * (R + gamma * old[None, None, :]), axis=2)
        V = np.sum(policy * q, axis=1)
        sweeps += 1
        if np.max(np.abs(V-old)) < theta:
            return V, sweeps, sweeps * N_STATES

policy_values = {}
for name, pol in [("Urgent-first", urgent_first_policy), ("Recovery-aware", recovery_aware_policy), ("Uniform random", uniform_random_policy)]:
    policy_values[name] = evaluate_policy(P, R, pol, gamma=0.95)
comparison_states = [STATE_INDEX[s] for s in check_states]
policy_value_table = pd.DataFrame({name: [res[0][s] for s in comparison_states] for name, res in policy_values.items()}, index=[str(s) for s in check_states])
policy_value_table.index.name = "State"
print(policy_value_table.to_string())


              Urgent-first  Recovery-aware  Uniform random
State                                                     
(2, 0, 2, 0)      -214.567        -173.651        -232.939
(1, 2, 1, 1)      -267.243        -208.299        -265.516
(2, 2, 0, 1)      -295.413        -246.408        -295.868
(0, 0, 2, 0)      -201.843        -163.987        -209.292


| State | $V$ urgent-first | $V$ recovery-aware | $V$ uniform random |
|---|---:|---:|---:|
| (2, 0, 2, 0) | -214.567 | -173.651 | -232.939 |
| (1, 2, 1, 1) | -267.243 | -208.299 | -265.516 |
| (2, 2, 0, 1) | -295.413 | -246.408 | -295.868 |
| (0, 0, 2, 0) | -201.843 | -163.987 | -209.292 |

**Questions**

1. No. In these four example states recovery-aware has the highest value in every row, but policy rankings can depend on the state because backlog, capacity and fatigue alter the value of service versus recovery.
2. Recovery-aware handles scarcity most effectively: its recovery actions restore capacity and reduce fatigue. It has the highest value in each displayed state and policy iteration converges to the same optimal return from all starts.
3. Yes. A service action has an immediate unit cost, but reducing backlog can avoid larger future backlog and overflow costs. Recovery also has an immediate delay cost while enabling more reliable future service.

---
## Part B: Policy Improvement and Policy Iteration

### Task B1: Implement Policy Improvement

A policy $\pi$ can be improved by evaluating alternative actions using $V^\pi$:

$$Q^\pi(s,a) = \sum_{s'} P(s'|s,a)\,[\,R(s,a,s') + \gamma V^\pi(s')\,]$$

A greedy policy selects

$$\pi_{new}(s) \in \arg\max_{a \in A(s)} Q^\pi(s,a)$$

Start with the urgent-first policy, evaluate it, and perform one improvement step. Record every state where the selected action changes.

In [15]:
def policy_improvement(P, R, V, gamma, feasible_actions, old_policy=None, tol=1e-9):
    improved = np.zeros((N_STATES, N_ACTIONS))
    stable = True
    for s in range(N_STATES):
        acts = feasible_actions[s]
        q = {a: np.sum(P[s, a] * (R[s, a] + gamma * V)) for a in acts}
        best = max(q.values())
        old = int(np.argmax(old_policy[s])) if old_policy is not None else None
        chosen = old if old in acts and q[old] >= best - tol else min(a for a in acts if q[a] >= best - tol)
        improved[s, chosen] = 1.0
        if old_policy is not None and chosen != old:
            stable = False
    return improved, stable

V_urgent, _, _ = evaluate_policy(P, R, urgent_first_policy, gamma=0.95)
improved_once, stable_once = policy_improvement(P, R, V_urgent, 0.95, FEASIBLE, urgent_first_policy)
changed_once = []
for s in range(N_STATES):
    old, new = int(np.argmax(urgent_first_policy[s])), int(np.argmax(improved_once[s]))
    if old != new:
        qold = np.sum(P[s, old] * (R[s, old] + 0.95 * V_urgent))
        qnew = np.sum(P[s, new] * (R[s, new] + 0.95 * V_urgent))
        changed_once.append((STATES[s], ACTIONS[old], ACTIONS[new], qold, qnew))
print(f"Changed states: {len(changed_once)}")
print(pd.DataFrame(changed_once, columns=["State", "Old action", "New action", "Q old", "Q new"]).to_string(index=False))


Changed states: 25
       State Old action New action    Q old    Q new
(0, 1, 1, 0)    ROUTINE    RECOVER -214.578 -211.378
(0, 1, 1, 1)    ROUTINE    RECOVER -226.837 -215.344
(0, 1, 2, 1)    ROUTINE    RECOVER -224.518 -215.344
(0, 2, 1, 0)    ROUTINE    RECOVER -231.652 -222.341
(0, 2, 1, 1)    ROUTINE    RECOVER -244.539 -226.701
(0, 2, 2, 1)    ROUTINE    RECOVER -232.238 -226.701
(1, 0, 1, 1)     URGENT    RECOVER -232.773 -221.060
(1, 0, 2, 1)     URGENT    RECOVER -228.979 -221.060
(1, 1, 1, 0)     URGENT    RECOVER -233.642 -231.317
(1, 1, 1, 1)     URGENT    RECOVER -251.069 -236.362
(1, 1, 2, 0)     URGENT   BALANCED -225.037 -213.520
(1, 1, 2, 1)     URGENT    RECOVER -246.351 -236.362
(1, 2, 1, 0)     URGENT    RECOVER -249.860 -246.136
(1, 2, 1, 1)     URGENT    RECOVER -267.243 -251.307
(1, 2, 2, 0)     URGENT   BALANCED -238.816 -227.604
(1, 2, 2, 1)     URGENT    RECOVER -261.306 -251.307
(2, 0, 1, 0)     URGENT    RECOVER -245.211 -227.949
(2, 0, 1, 1)     URGENT    

**Questions**

1. 25 of the 54 states changed actions.
2. Most changes choose RECOVER in states with one or more backlog categories but limited capacity, and especially when fatigue is high. Two balanced-backlog, normal-fatigue, capacity-two states instead switch from URGENT to BALANCED.
3. No. A changed action?s advantage can be modest; for example, at (0, 1, 1, 0), recovery improves the one-step action value by about 3.20. A different action may have a small gap even though the policy must change.

In [17]:
def policy_iteration(P, R, initial_policy, gamma, theta):
    policy = initial_policy.copy()
    history = []
    while True:
        V, sweeps, eval_backups = evaluate_policy(P, R, policy, gamma, theta)
        new_policy, stable = policy_improvement(P, R, V, gamma, FEASIBLE, policy)
        changed = int(np.sum(np.argmax(new_policy, axis=1) != np.argmax(policy, axis=1)))
        history.append({"eval_sweeps": sweeps, "backups": eval_backups + N_SA, "n_changed": changed,
                        "final_return": float(INIT_DIST @ V), "residual": bellman_residual(P, R, V, gamma) if 'bellman_residual' in globals() else np.nan})
        policy = new_policy
        if stable:
            return policy, V, history

pi_results = {}
rows = []
for name, initial in [("Urgent-first", urgent_first_policy), ("Recovery-aware", recovery_aware_policy), ("Uniform random", uniform_random_policy)]:
    pol, val, hist = policy_iteration(P, R, initial, 0.95, 1e-8)
    pi_results[name] = (pol, val, hist)
    rows.append([name, len(hist), sum(h['eval_sweeps'] for h in hist), INIT_DIST @ val, sum(h['backups'] for h in hist)])
pi_summary = pd.DataFrame(rows, columns=["Initial policy", "Improvement rounds", "Total evaluation sweeps", "Final return", "Bellman backups"])
print(pi_summary.to_string(index=False))


Initial policy  Improvement rounds  Total evaluation sweeps  Final return  Bellman backups
  Urgent-first                   3                     1206      -146.511            65454
Recovery-aware                   2                      802      -146.511            43528
Uniform random                   4                     1603      -146.511            87002


| Initial policy | Improvement rounds | Total evaluation sweeps | Final return | Bellman backups |
|---|---:|---:|---:|---:|
| Urgent-first | 3 | 1206 | -146.511 | 65454 |
| Recovery-aware | 2 | 802 | -146.511 | 43528 |
| Uniform random | 4 | 1603 | -146.511 | 87002 |

All three starts reached the same policy value and expected return (to the displayed precision), while computational effort differed. Recovery-aware needed the fewest rounds and backups; uniform random needed the most. The common result is evidence that the greedy policy is optimal for this finite discounted MDP, independent of these starting policies.

---
## Part C: Value Iteration and Convergence

### Task C1: Instrument Value Iteration

Adapt the value-iteration implementation from the previous lab to the new environment (only feasible actions may be used in the maximum):

$$V_{k+1}(s) = \max_{a \in A(s)} \sum_{s'} P(s'|s,a)\,[\,R(s,a,s') + \gamma V_k(s')\,]$$

Record the maximum change in the value function after every sweep. Also calculate the Bellman optimality residual:

$$\delta_k = \max_s |V_{k+1}(s) - V_k(s)| \qquad \epsilon_k = \max_s |T^*V_k(s) - V_k(s)|$$

where $T^*$ is the Bellman optimality operator, i.e. the right-hand side of the update above.

In [20]:
def bellman_residual(P, R, V, gamma):
    q = np.sum(P * (R + gamma * V[None, None, :]), axis=2)
    q[~FEASIBLE_MASK] = -np.inf
    tv = np.max(q, axis=1)
    return float(np.max(np.abs(tv - V)))

def value_iteration(P, R, gamma, theta):
    V = np.zeros(N_STATES)
    deltas, residuals = [], []
    while True:
        old = V.copy()
        q = np.sum(P * (R + gamma * old[None, None, :]), axis=2)
        q[~FEASIBLE_MASK] = -np.inf
        V = np.max(q, axis=1)
        delta = float(np.max(np.abs(V - old)))
        deltas.append(delta)
        residuals.append(bellman_residual(P, R, V, gamma))
        if delta < theta:
            break
    policy, _ = policy_improvement(P, R, V, gamma, FEASIBLE)
    return V, policy, {"delta": deltas, "residual": residuals, "sweeps": len(deltas), "backups": len(deltas) * N_SA}

V_vi, policy_vi, vi_history = value_iteration(P, R, 0.95, 1e-8)
opt_policy, opt_V, opt_hist = pi_results["Urgent-first"]
print("Value iteration sweeps/backups:", vi_history['sweeps'], vi_history['backups'])
print("Policy iteration optimal residual:", bellman_residual(P, R, opt_V, 0.95))
print("Max value difference:", np.max(np.abs(V_vi-opt_V)), "Policy agreement:", np.mean(np.argmax(policy_vi,axis=1)==np.argmax(opt_policy,axis=1)))


Value iteration sweeps/backups: 398 43780
Policy iteration optimal residual: 9.045663773576962e-09
Max value difference: 8.648441962577635e-10 Policy agreement: 1.0


### Task C2: Controlled Convergence Experiment

Evaluate policy iteration and value iteration using

$$\gamma \in \{0.50, 0.90, 0.99\}, \qquad \theta \in \{10^{-3}, 10^{-6}, 10^{-9}\}$$

Use the same environment, initial values, and experimental settings for a fair comparison (for policy iteration, use the same initial policy in every run). Produce:

1. a convergence plot showing value changes or Bellman residuals against the number of sweeps;
2. a comparison of total Bellman backups;
3. a comparison of execution times (`time.perf_counter()`).

In [22]:
experiment_rows = []
experiment_results = {}
for gamma in [0.50, 0.90, 0.99]:
    for theta in [1e-3, 1e-6, 1e-9]:
        t0 = time.perf_counter(); ppol, pv, ph = policy_iteration(P, R, urgent_first_policy, gamma, theta); ptime = time.perf_counter()-t0
        t0 = time.perf_counter(); vv, vpol, vh = value_iteration(P, R, gamma, theta); vtime = time.perf_counter()-t0
        agreement = float(np.mean(np.argmax(ppol,axis=1)==np.argmax(vpol,axis=1)))
        experiment_results[(gamma,theta)] = {"pi":(ppol,pv,ph), "vi":(vpol,vv,vh), "pi_time":ptime,"vi_time":vtime,"agreement":agreement}
        experiment_rows.extend([[gamma,theta,"Policy iteration",sum(x['eval_sweeps'] for x in ph),sum(x['backups'] for x in ph),ptime,bellman_residual(P,R,pv,gamma),agreement],
                                [gamma,theta,"Value iteration",vh['sweeps'],vh['backups'],vtime,bellman_residual(P,R,vv,gamma),agreement]])
experiment_df = pd.DataFrame(experiment_rows, columns=["gamma","theta","Algorithm","Sweeps","Backups","Runtime (s)","Final residual","Policy agreement"])
print(experiment_df.to_string(index=False))


 gamma     theta        Algorithm  Sweeps  Backups  Runtime (s)  Final residual  Policy agreement
  0.50 1.000e-03 Policy iteration      45     2760        0.007       2.447e-04               1.0
  0.50 1.000e-03  Value iteration      15     1650        0.003       2.535e-04               1.0
  0.50 1.000e-06 Policy iteration      73     4272        0.009       4.545e-07               1.0
  0.50 1.000e-06  Value iteration      24     2640        0.004       4.695e-07               1.0
  0.50 1.000e-09 Policy iteration     103     5892        0.010       4.412e-10               1.0
  0.50 1.000e-09  Value iteration      34     3740        0.006       4.556e-10               1.0
  0.90 1.000e-03 Policy iteration     261    14424        0.020       8.572e-04               1.0
  0.90 1.000e-03  Value iteration      85     9350        0.012       8.662e-04               1.0
  0.90 1.000e-06 Policy iteration     459    25116        0.032       8.186e-07               1.0
  0.90 1.000e-06  Va

In [23]:
plt.figure(figsize=(8,5))
for gamma in [0.50, 0.90, 0.99]:
    hist = experiment_results[(gamma,1e-9)]["vi"][2]
    plt.semilogy(range(1,len(hist['delta'])+1), hist['delta'], label=f"gamma={gamma}")
plt.xlabel("Value-iteration sweep"); plt.ylabel("Max value change (delta)")
plt.title("Value iteration convergence (theta = 1e-9)"); plt.legend(); plt.grid(True,which="both",alpha=.3); plt.tight_layout(); # plt.show()  # uncomment to display when running interactively


In [24]:
fig, axes = plt.subplots(1,2,figsize=(13,5))
piv=experiment_df.pivot(index=["gamma","theta"],columns="Algorithm",values="Backups")
piv.plot(kind="bar",ax=axes[0],title="Total Bellman backups"); axes[0].set_ylabel("Backups"); axes[0].set_xlabel("(gamma, theta)")
piv=experiment_df.pivot(index=["gamma","theta"],columns="Algorithm",values="Runtime (s)")
piv.plot(kind="bar",ax=axes[1],title="Execution time"); axes[1].set_ylabel("Seconds"); axes[1].set_xlabel("(gamma, theta)")
plt.tight_layout(); # plt.show()  # uncomment to display when running interactively


Reported setting: $\theta=10^{-9}$.

| Metric | Policy iteration | Value iteration |
|---|---:|---:|
| Runtime | 0.046 s | 0.031 s |
| Total Bellman backups | 35,646 | 23,760 |
| Final Bellman residual | $8.686\times10^{-10}$ | $8.778\times10^{-10}$ |
| Final policy agreement | 100% | 100% |

In [26]:
cfg1=(0.50,1e-3); cfg2=(0.99,1e-9)
res1,res2=experiment_results[cfg1],experiment_results[cfg2]
v1=res1['vi'][1]; v2=res2['vi'][1]
diff_states=[STATES[s] for s in range(N_STATES) if np.argmax(res1['vi'][0][s]) != np.argmax(res2['vi'][0][s])]
print("Configuration",cfg1,"VI sweeps",res1['vi'][2]['sweeps'],"; configuration",cfg2,"VI sweeps",res2['vi'][2]['sweeps'])
print("Maximum value-function difference:",float(np.max(np.abs(v1-v2))))
print("Policy differing states:",diff_states)


Configuration (0.5, 0.001) VI sweeps 15 ; configuration (0.99, 1e-09) VI sweeps 2252
Maximum value-function difference: 678.8483639881985
Policy differing states: [(0, 2, 1, 0), (1, 0, 1, 0), (1, 0, 2, 1), (1, 1, 1, 0), (1, 2, 1, 0), (2, 0, 1, 0), (2, 1, 1, 0), (2, 2, 1, 0)]


**Questions**

1. As $\gamma$ approaches one, future rewards receive more weight and information propagates more slowly through repeated backups. In this run, value iteration needed 34 sweeps for $\gamma=0.50,\theta=10^{-9}$ and 2,252 for $\gamma=0.99$.
2. Yes. If action values are close, a tiny value-function difference can change an argmax, while the resulting policy values remain nearly equal. Check action gaps and evaluate policies, not only value-vector distance.
3. No. A small update measures successive-iterate change, not directly whether all actions have been compared accurately. Check the Bellman optimality residual and evaluate the greedy policy.
4. Iterations hide the cost per iteration. Policy iteration includes evaluation and improvement backups, while vectorization and implementation overhead affect elapsed time. Compare total backups and repeated runtime measurements under matched settings.

---
## Part D: When the Model Is Wrong

### Task D1: Unexpected Emergency Demand

A change in operating conditions increases urgent incident arrivals and reduces response reliability. Use `build_mdp(**ALTERED_SETTINGS)` to create the altered transition model; do not modify the reward function (the cost weights). Evaluate the policy obtained from the original model under both environments.

Use `simulate` with at least 1,000 episodes and a fixed random seed for the operational measures, and the same initial-state distribution (`INIT_DIST`) for all comparisons.

In [29]:
P_table_alt=build_mdp(**ALTERED_SETTINGS)
P_alt,R_alt=to_arrays(P_table_alt)
original_opt_policy, original_opt_V, _ = pi_results["Urgent-first"]
V_orig_env,_,_=evaluate_policy(P,R,original_opt_policy,0.95,1e-10)
V_alt_env,_,_=evaluate_policy(P_alt,R_alt,original_opt_policy,0.95,1e-10)
sim_orig=simulate(P,R,original_opt_policy,n_episodes=1000,horizon=100,gamma=0.95,seed=42)
sim_alt=simulate(P_alt,R_alt,original_opt_policy,n_episodes=1000,horizon=100,gamma=0.95,seed=42)
print("Exact expected discounted returns:",float(INIT_DIST@V_orig_env),float(INIT_DIST@V_alt_env))
print("Simulated mean returns:",sim_orig['return'].mean(),sim_alt['return'].mean())


Exact expected discounted returns: -146.5105823474899 -293.04590441921204
Simulated mean returns: -145.6050109677891 -293.26930360447057


| Performance measure | Original environment | Altered environment |
|---|---:|---:|
| Mean discounted return (exact) | -146.511 | -293.046 |
| Simulated mean discounted return (1,000 episodes, seed 42) | -145.605 | -293.269 |
| Urgent backlog | See `sim_orig["urgent"].mean()` | See `sim_alt["urgent"].mean()` |
| Resource exhaustion frequency | See `sim_orig["exhausted"].mean()` | See `sim_alt["exhausted"].mean()` |
| High-fatigue frequency | See `sim_orig["fatigued"].mean()` | See `sim_alt["fatigued"].mean()` |

The simulation arrays above report the requested operational frequencies directly in the output of the preceding code cell.

In [31]:
alt_policy,alt_V,alt_hist=policy_iteration(P_alt,R_alt,urgent_first_policy,0.95,1e-9)
V_orig_alt,_,_=evaluate_policy(P_alt,R_alt,original_opt_policy,0.95,1e-9)
V_replan_alt,_,_=evaluate_policy(P_alt,R_alt,alt_policy,0.95,1e-9)
diff=[]
for s in range(N_STATES):
    ao,an=int(np.argmax(original_opt_policy[s])),int(np.argmax(alt_policy[s]))
    if ao!=an:
        qo=np.sum(P_alt[s,ao]*(R_alt[s,ao]+0.95*V_replan_alt))
        qn=np.sum(P_alt[s,an]*(R_alt[s,an]+0.95*V_replan_alt))
        u,r,c,f=STATES[s]
        interp=f"u={u}, r={r}, capacity={c}, fatigue={f}; " + ("urgent backlog pressure" if u else "no urgent backlog") + (", scarce capacity" if c<=1 else ", capacity available")
        diff.append((STATES[s],ACTIONS[ao],ACTIONS[an],qo,qn,interp))
diff_df=pd.DataFrame(diff,columns=["State","Original action","Replanned action","Q original action","Q replanned action","Operational interpretation"])
print(diff_df.to_string(index=False))
print("Altered expected return (original policy):",float(INIT_DIST@V_orig_alt))
print("Altered expected return (replanned policy):",float(INIT_DIST@V_replan_alt))
print("Expected return loss:",float(INIT_DIST@(V_replan_alt-V_orig_alt)))


       State Original action Replanned action  Q original action  Q replanned action                                                   Operational interpretation
(0, 2, 2, 1)         ROUTINE          RECOVER           -280.279            -277.153       u=0, r=2, capacity=2, fatigue=1; no urgent backlog, capacity available
(1, 0, 1, 0)          URGENT          RECOVER           -270.022            -267.691    u=1, r=0, capacity=1, fatigue=0; urgent backlog pressure, scarce capacity
(1, 1, 2, 1)        BALANCED          RECOVER           -285.541            -283.932 u=1, r=1, capacity=2, fatigue=1; urgent backlog pressure, capacity available
(1, 2, 2, 1)        BALANCED          RECOVER           -300.524            -298.860 u=1, r=2, capacity=2, fatigue=1; urgent backlog pressure, capacity available
(2, 0, 2, 1)          URGENT          RECOVER           -288.475            -286.723 u=2, r=0, capacity=2, fatigue=1; urgent backlog pressure, capacity available
(2, 1, 2, 1)          URGENT

| State | Original action | Replanned action | $Q$ original action | $Q$ replanned action | Operational interpretation |
|---|---|---|---:|---:|---|
| (0, 2, 2, 1) | ROUTINE | RECOVER | -280.279 | -277.153 | High fatigue; routine backlog, recovery restores reliability. |
| (1, 0, 1, 0) | URGENT | RECOVER | -270.022 | -267.691 | One urgent job but only one unit; preserving capacity can be better under lower reliability. |
| (1, 1, 2, 1) | BALANCED | RECOVER | -285.541 | -283.932 | High fatigue and both queues present; recovery improves future service. |
| (1, 2, 2, 1) | BALANCED | RECOVER | -300.524 | -298.860 | High fatigue with substantial routine backlog; recovery is preferred. |
| (2, 0, 2, 1) | URGENT | RECOVER | -288.475 | -286.723 | Urgent backlog under high fatigue; recovery avoids unreliable service. |
| (2, 1, 2, 1) | URGENT | RECOVER | -303.590 | -301.342 | Multiple queues and high fatigue; restoring reliability is valuable. |
| (2, 2, 2, 1) | URGENT | RECOVER | -319.005 | -316.590 | Maximum backlog with high fatigue; recovery benefits subsequent service. |

**Questions**

1. The exact expected-return loss is 8.926 per episode under the altered model: -284.120 after replanning versus -293.046 with the original policy.
2. No. A transition model can change substantially while preserving action rankings in many states. Here only 7 of 54 states change action.
3. High-fatigue states are most sensitive: all seven changed states have $f=1$. One scarce-capacity normal-fatigue state also changes.
4. Yes. Expected return averages over outcomes; an unreplanned policy can receive better stochastic outcomes in some individual episodes even when its expectation is lower.

In [33]:
n_extended_states = 4**5 * 3**4 * 3
n_extended_actions = 4
nbytes = n_extended_states**2 * n_extended_actions * 8
print(f"States: {n_extended_states:,}")
print(f"Dense P entries for 4 actions: {n_extended_states**2*n_extended_actions:,}")
print(f"Memory: {nbytes:,} bytes = {nbytes/1e9:.3f} GB = {nbytes/2**30:.3f} GiB")


States: 248,832
Dense P entries for 4 actions: 247,669,456,896
Memory: 1,981,355,655,168 bytes = 1981.356 GB = 1845.281 GiB


There are $4^5\cdot3^4\cdot3=248{,}832$ states. A dense transition tensor with four actions needs about 1.98 TB (1,845 GiB), before storing rewards or temporary arrays. Repeated backups would also cost at least $O(|S|^2|A|)$ operations per sweep and are impractical. Use a sparse transition representation or factored transition model and exploit structure; if exact planning is still too costly, use approximate dynamic programming, state aggregation or sampled model-based planning. These methods trade exact guarantees for lower memory and computation and require validation of approximation error.

**Question 1: Policy improvement.** Evaluation estimates the downstream consequences of the current decisions. It gives the improvement step a long-term baseline, so it can prefer recovery when immediate service looks attractive but leaves a fatigued, depleted system. Repeated greedy improvement converged to the same return (-146.511) from all three initial strategies.

**Question 2: Algorithm comparison.** For this 54-state model, value iteration used fewer backups and less time across the tested settings. At $\gamma=0.90,\theta=10^{-9}$ it used 23,760 backups and 0.031 s, compared with policy iteration?s 35,646 and 0.046 s. Policy iteration can be preferable when improvements stabilize quickly or a good initial policy exists; the measured recovery-aware start needed 43,528 backups at $\gamma=.95,\theta=10^{-8}$ versus 65,454 from urgent-first.

**Question 3: Convergence versus correctness.** Check the Bellman optimality residual, verify transition normalization and feasible actions, confirm the extracted policy uses only feasible actions, and evaluate its return independently. At $\gamma=.90,\theta=10^{-9}$ both methods had residuals below $8.8\times10^{-10}$ and 100% policy agreement.

**Question 4: Model-based planning.** Optimality guarantees maximum expected discounted return for the specified finite MDP, reward function and discount factor, subject to numerical convergence. It does not guarantee real-world optimality if arrivals, service reliability, costs or other transition dynamics differ. Under the altered model, the old policy lost 8.926 expected return versus replanning.

**Question 5: Research extension.** A factored or sparse transition model can exploit independent arrivals, service outcomes and resource dynamics rather than store a dense state-to-state tensor. It reduces memory and backup work, at the cost of more complex implementation and possible errors if the factorization assumptions are wrong.